In [2]:
# # -*- coding: utf-8 -*-
# """
# BATCH PROCESSING FOR SARCASM DETECTION
# This script performs sarcasm detection on hotel reviews using batch processing.
# """

# # =============================================================================
# # INSTALLATION AND IMPORTS
# # =============================================================================
# print("Installing required packages...")
# !pip install pandas numpy
# !pip install emot langdetect

# import pandas as pd
# import numpy as np
# import re
# import time
# import json
# import warnings
# import requests
# from typing import List, Dict, Any
# warnings.filterwarnings('ignore')

# # Import for preprocessing
# from langdetect import detect, DetectorFactory, LangDetectException
# from langdetect import detect_langs
# from emot.emo_unicode import EMOTICONS_EMO

# # Set random seeds for reproducibility
# DetectorFactory.seed = 0
# np.random.seed(42)

# # =============================================================================
# # CONFIGURATION
# # =============================================================================
# # File paths (adjust based on your Google Drive structure)
# INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels.xlsx"
# OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels_sarcasm_batch_processed.xlsx"

# # API settings (using a free API as fallback)
# API_URL = "https://api-inference.huggingface.co/models/cardiffnlp/twitter-roberta-base-sarcasm"
# API_TOKEN = "your_huggingface_token_here"  # Replace with your token

# # Processing settings
# BATCH_SIZE = 5  # Smaller batch size for API limits
# DELAY_BETWEEN_BATCHES = 2  # Delay between batch requests in seconds
# REVIEW_COLUMN = "Review Summary"  # Column containing review text
# SAMPLE_SIZE = 200  # Set to None for full dataset, or number for testing

# # =============================================================================
# # GOOGLE DRIVE SETUP
# # =============================================================================
# print("Mounting Google Drive...")
# from google.colab import drive
# drive.mount('/content/drive')

# # =============================================================================
# # PREPROCESSING FUNCTIONS
# # =============================================================================
# def handle_emoticons(text):
#     """
#     Convert emoticons to text descriptions.
#     Important for sarcasm analysis.
#     Example: :) → "smiling face"
#     """
#     if not hasattr(handle_emoticons, 'emoticon_patterns'):
#         handle_emoticons.emoticon_patterns = []
#         for emoticon, description in EMOTICONS_EMO.items():
#             escaped_emoticon = re.escape(emoticon)
#             handle_emoticons.emoticon_patterns.append(
#                 (re.compile(escaped_emoticon), f" {description} ")
#             )

#     text = str(text)
#     for pattern, replacement in handle_emoticons.emoticon_patterns:
#         text = pattern.sub(replacement, text)
#     return text

# def is_english_reliable(text):
#     """
#     Detect if text is English with high confidence.
#     Filters out non-English reviews to improve analysis quality.
#     """
#     if pd.isna(text) or text == "" or len(str(text).strip()) < 10:
#         return False

#     text = str(text)

#     try:
#         text_sample = text[:500].strip()
#         lang_probabilities = detect_langs(text_sample)

#         for lang_prob in lang_probabilities:
#             if lang_prob.lang == 'en' and lang_prob.prob >= 0.7:
#                 return True
#             elif lang_prob.lang == 'en' and lang_prob.prob >= 0.5 and len(text_sample) > 50:
#                 return True

#         return False

#     except (LangDetectException, Exception):
#         text_lower = text.lower()
#         english_words = {'the', 'and', 'to', 'of', 'a', 'in', 'that', 'is', 'it', 'for', 'was', 'but', 'with', 'on', 'he', 'she', 'they', 'we', 'you', 'i'}
#         words = re.findall(r'\b[a-z]+\b', text_lower)

#         if not words or len(words) < 5:
#             return False

#         english_count = sum(1 for word in words if word in english_words)
#         return (english_count / len(words)) > 0.3

# def preprocess_for_sarcasm(text):
#     """
#     Specialized preprocessing for sarcasm detection.
#     Preserves case, punctuation, and linguistic cues important for sarcasm.
#     """
#     if pd.isna(text) or text == "":
#         return ""

#     text = str(text)

#     # Basic cleaning only - preserve case and punctuation!
#     text = re.sub(r'http\S+|www\.\S+', '', text)
#     text = re.sub(r'\S+@\S+', '', text)
#     text = re.sub(r'\s+', ' ', text)

#     # Handle emoticons (crucial for sarcasm)
#     text = handle_emoticons(text)

#     return text.strip()

# # =============================================================================
# # BATCH SARCASM DETECTION MODULE
# # =============================================================================
# class BatchSarcasmDetector:
#     """API-based sarcasm detection using batch processing"""

#     def __init__(self, batch_size=BATCH_SIZE):
#         self.batch_size = batch_size
#         self.headers = {"Authorization": f"Bearer {API_TOKEN}"} if API_TOKEN != "your_huggingface_token_here" else {}

#     def query_api(self, payload):
#         """Query the HuggingFace API for sarcasm detection"""
#         try:
#             response = requests.post(API_URL, headers=self.headers, json=payload)
#             return response.json()
#         except Exception as e:
#             print(f"API request failed: {e}")
#             return None

#     def analyze_batch(self, reviews_batch: List[str]) -> List[Dict[str, Any]]:
#         """Analyze a batch of reviews using the API"""
#         if not reviews_batch:
#             return []

#         # Prepare payload for the API
#         payload = {"inputs": reviews_batch}

#         try:
#             response = self.query_api(payload)

#             if response and isinstance(response, list):
#                 results = []
#                 for item in response:
#                     if isinstance(item, list) and len(item) > 0:
#                         # Get the first (and usually only) result for each review
#                         result = item[0]
#                         sarcasm_score = result.get('score', 0) if result.get('label', '') == 'sarcasm' else 0

#                         results.append({
#                             'sarcasm': sarcasm_score > 0.5,
#                             'confidence': sarcasm_score,
#                             'reason': 'API-based detection',
#                             'type': 'positive_sarcasm' if sarcasm_score > 0.5 else 'none'
#                         })
#                     else:
#                         # Fallback if response format is unexpected
#                         results.append(self.heuristic_fallback(""))

#                 return results
#             else:
#                 # Fallback if API fails
#                 return [self.heuristic_fallback(review) for review in reviews_batch]

#         except Exception as e:
#             print(f"Error processing batch: {e}")
#             return [self.heuristic_fallback(review) for review in reviews_batch]

#     def heuristic_fallback(self, text):
#         """Fallback method when API analysis fails"""
#         text_lower = text.lower() if text else ""
#         sarcastic_keywords = [
#             'sarcastic', 'ironic', 'obviously', 'of course', 'surprise',
#             'wow', 'really', 'just what i needed', 'thanks for', 'perfect',
#             'as expected', 'big surprise', 'shockingly', 'unbelievably'
#         ]

#         # More sophisticated heuristic checks
#         is_sarcastic = any(kw in text_lower for kw in sarcastic_keywords)

#         # Check for exaggerated praise in short reviews
#         if not is_sarcastic and text_lower:
#             positive_words = ['excellent', 'amazing', 'perfect', 'fantastic', 'outstanding', 'wonderful']
#             has_exaggeration = sum(1 for word in positive_words if word in text_lower) >= 2
#             is_short = len(text_lower.split()) < 10
#             is_sarcastic = has_exaggeration and is_short

#         # Check for negative words in positive context
#         if not is_sarcastic and text_lower:
#             negative_words = ['terrible', 'awful', 'horrible', 'disappointing', 'bad']
#             has_negative = any(word in text_lower for word in negative_words)
#             has_positive = any(word in text_lower for word in positive_words)
#             is_sarcastic = has_negative and has_positive

#         confidence = 0.7 if is_sarcastic else 0.3

#         return {
#             'sarcasm': is_sarcastic,
#             'confidence': confidence,
#             'reason': 'Heuristic fallback detection',
#             'type': 'positive_sarcasm' if is_sarcastic else 'none'
#         }

#     def process_dataframe(self, df, review_column=REVIEW_COLUMN):
#         """Process entire dataframe using batch processing"""
#         print(f"Processing {len(df)} reviews for sarcasm using batch size {self.batch_size}...")

#         df_sarcasm = df.copy()
#         df_sarcasm['sarcasm'] = 0  # 0 = not sarcastic, 1 = sarcastic
#         df_sarcasm['sarcasm_confidence'] = 0.0
#         df_sarcasm['sarcasm_reason'] = 'Not processed'
#         df_sarcasm['sarcasm_type'] = 'none'
#         df_sarcasm['processed_review'] = df_sarcasm[review_column].apply(preprocess_for_sarcasm)

#         # Filter out empty reviews
#         valid_indices = []
#         valid_reviews = []

#         for idx, row in df_sarcasm.iterrows():
#             if pd.isna(row['processed_review']) or row['processed_review'].strip() == '':
#                 # Apply fallback to empty reviews immediately
#                 result = self.heuristic_fallback("")
#                 df_sarcasm.at[idx, 'sarcasm'] = 1 if result.get('sarcasm', False) else 0
#                 df_sarcasm.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
#                 df_sarcasm.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
#                 df_sarcasm.at[idx, 'sarcasm_type'] = result.get('type', 'none')
#             else:
#                 valid_indices.append(idx)
#                 valid_reviews.append(row['processed_review'])

#         # Process valid reviews in batches
#         total_batches = (len(valid_reviews) + self.batch_size - 1) // self.batch_size
#         batch_count = 0

#         for i in range(0, len(valid_reviews), self.batch_size):
#             batch_count += 1
#             batch_indices = valid_indices[i:i+self.batch_size]
#             batch_reviews = valid_reviews[i:i+self.batch_size]

#             print(f"Processing batch {batch_count}/{total_batches} ({len(batch_reviews)} reviews)")

#             try:
#                 batch_results = self.analyze_batch(batch_reviews)

#                 # Apply results to dataframe
#                 for idx, result in zip(batch_indices, batch_results):
#                     df_sarcasm.at[idx, 'sarcasm'] = 1 if result.get('sarcasm', False) else 0
#                     df_sarcasm.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
#                     df_sarcasm.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
#                     df_sarcasm.at[idx, 'sarcasm_type'] = result.get('type', 'none')

#             except Exception as e:
#                 print(f"Error processing batch {batch_count}: {e}")
#                 # Apply fallback to all reviews in this batch
#                 for idx in batch_indices:
#                     result = self.heuristic_fallback(df_sarcasm.at[idx, 'processed_review'])
#                     df_sarcasm.at[idx, 'sarcasm'] = 1 if result.get('sarcasm', False) else 0
#                     df_sarcasm.at[idx, 'sarcasm_confidence'] = result.get('confidence', 0.0)
#                     df_sarcasm.at[idx, 'sarcasm_reason'] = str(result.get('reason', ''))[:200]
#                     df_sarcasm.at[idx, 'sarcasm_type'] = result.get('type', 'none')

#             # Delay between batches to avoid overwhelming the API
#             if batch_count < total_batches:
#                 time.sleep(DELAY_BETWEEN_BATCHES)

#         sarcastic_count = df_sarcasm['sarcasm'].sum()
#         print(f"Sarcasm detection completed! Sarcastic reviews: {sarcastic_count}/{len(df_sarcasm)}")

#         return df_sarcasm

# # =============================================================================
# # MAIN EXECUTION
# # =============================================================================
# def main():
#     """
#     Main function that runs batch sarcasm detection on hotel reviews.
#     """
#     print("="*60)
#     print("BATCH SARCASM DETECTION PIPELINE")
#     print("="*60)

#     # Step 1: Load data
#     print("1. Loading data...")
#     try:
#         df = pd.read_excel(INPUT_FILE)
#         print(f"   Loaded {len(df)} reviews from {INPUT_FILE}")
#     except Exception as e:
#         print(f"   Error loading file: {e}")
#         return None

#     # For testing, use a subset
#     if SAMPLE_SIZE and SAMPLE_SIZE < len(df):
#         print(f"   Using sample of {SAMPLE_SIZE} reviews for testing")
#         df = df.head(SAMPLE_SIZE).copy()

#     # Step 2: Filter English reviews
#     print("2. Filtering English reviews...")
#     initial_count = len(df)
#     english_mask = df[REVIEW_COLUMN].apply(is_english_reliable)
#     df = df[english_mask]
#     print(f"   Removed {initial_count - len(df)} non-English reviews")

#     if len(df) == 0:
#         print("   No English reviews found!")
#         return None

#     # Step 3: Run batch sarcasm detection
#     print("3. Running batch sarcasm detection...")
#     detector = BatchSarcasmDetector(batch_size=BATCH_SIZE)
#     df_result = detector.process_dataframe(df)

#     # Step 4: Save results
#     print("4. Saving results...")
#     df_result.to_excel(OUTPUT_FILE, index=False)
#     print(f"   Results saved to {OUTPUT_FILE}")

#     # Step 5: Display summary
#     print("\n" + "="*60)
#     print("ANALYSIS SUMMARY")
#     print("="*60)
#     print(f"Total reviews analyzed: {len(df_result)}")

#     if 'sarcasm' in df_result.columns:
#         sarcastic = df_result['sarcasm'].sum()
#         print(f"Sarcastic reviews detected: {sarcastic} ({sarcastic/len(df_result)*100:.1f}%)")

#         # Show some examples
#         print("\nSample of sarcastic reviews:")
#         sarcastic_samples = df_result[df_result['sarcasm'] == 1].head(3)
#         for _, row in sarcastic_samples.iterrows():
#             print(f"  - {row[REVIEW_COLUMN][:100]}... (Confidence: {row['sarcasm_confidence']:.2f})")

#     return df_result

# # =============================================================================
# # EXECUTION
# # =============================================================================
# if __name__ == "__main__":
#     print("Starting batch sarcasm detection pipeline...")
#     results = main()

#     if results is not None:
#         print("\n✅ Analysis completed successfully!")
#         print(f"Final results contain {len(results)} reviews")
#     else:
#         print("\n❌ Analysis failed!")






# -*- coding: utf-8 -*-
"""
Optimized Batch Sarcasm Detection Pipeline (ML lifecycle)
- Data load & QA
- Preprocessing & language filtering
- (Optional) Weak-labeling via HuggingFace API for a small seed
- Feature extraction (TF-IDF)
- Model training (LogisticRegression) with GridSearchCV (regularization)
- Evaluation and model persistence
- Fast batch inference for entire dataset (no API per row)
- Save results to Excel
"""

# =============================================================================
# INSTALLS (uncomment when needed in Colab)
# =============================================================================
# !pip install pandas numpy joblib requests scikit-learn openpyxl emot langdetect

# =============================================================================
# IMPORTS & SEEDING
# =============================================================================
import os
import time
import re
import json
import math
import joblib
import logging
from typing import List, Dict, Any, Optional

import pandas as pd
import numpy as np
import requests
from emot.emo_unicode import EMOTICONS_EMO
from langdetect import detect_langs, DetectorFactory, LangDetectException

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# deterministic behavior
DetectorFactory.seed = 0
np.random.seed(42)

# =============================================================================
# CONFIGURATION
# =============================================================================
INPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels.xlsx"  # adjust
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/hotels_sarcasm_batch_processed.xlsx"
MODEL_FILE = "/content/drive/MyDrive/Colab Notebooks/sarcasm_model.joblib"
VECT_FILE = "/content/drive/MyDrive/Colab Notebooks/tfidf_vectorizer.joblib"

API_URL = "https://api-inference.huggingface.co/models/cardiffnlp/twitter-roberta-base-sarcasm"
API_TOKEN = "your_huggingface_token_here"  # replace with token or leave placeholder to skip seed labeling

BATCH_SIZE = 32                # used when calling API for seed labeling (not for final model inference)
API_DELAY = 1.0                # seconds between API calls (make larger if rate-limited)
SEED_LABEL_COUNT = 400         # how many samples to weak-label via API (optional)
REVIEW_COLUMN = "Review Summary"  # column in input Excel with review text
LABEL_COLUMN = "sarcasm_label"  # optional existing label in input; if present, will be used
SAMPLE_SIZE = None             # set integer for quick dev runs, or None for full data

LOGGING_LEVEL = logging.INFO

# =============================================================================
# LOGGING
# =============================================================================
logging.basicConfig(level=LOGGING_LEVEL, format="%(asctime)s - %(levelname)s - %(message)s")

# =============================================================================
# GOOGLE DRIVE (if using Colab) - keep, as in original
# =============================================================================
try:
    from google.colab import drive
    logging.info("Mounting Google Drive...")
    drive.mount('/content/drive')
except Exception:
    logging.info("Not in Colab or drive mount failed; continuing without mounting.")

# =============================================================================
# PREPROCESSING UTILITIES
# =============================================================================
# Build emoticon replacement patterns once
EMOTICON_PATTERNS = []
for emoticon, desc in EMOTICONS_EMO.items():
    # replace emoticon with short descriptor, keep spaces for tokenization
    EMOTICON_PATTERNS.append((re.compile(re.escape(emoticon)), f" {desc} "))

def handle_emoticons(text: str) -> str:
    s = str(text)
    for pattern, repl in EMOTICON_PATTERNS:
        s = pattern.sub(repl, s)
    return s

def is_english_reliable(text: str) -> bool:
    """Return True if detect_langs suggests English with decent confidence OR fallback heuristic."""
    if pd.isna(text) or not str(text).strip() or len(str(text).strip()) < 10:
        return False
    s = str(text).strip()[:500]
    try:
        langs = detect_langs(s)
        for lang_prob in langs:
            if lang_prob.lang == "en" and (lang_prob.prob >= 0.7 or (lang_prob.prob >= 0.5 and len(s) > 50)):
                return True
        return False
    except LangDetectException:
        # fallback: small heuristic using common English words
        words = re.findall(r'\b[a-zA-Z]{2,}\b', s.lower())
        if len(words) < 5:
            return False
        common = {'the','and','to','of','a','in','that','is','it','for'}
        common_count = sum(1 for w in words if w in common)
        return (common_count / len(words)) > 0.25

def preprocess_for_sarcasm(text: Any) -> str:
    """Light preprocessing: remove urls/emails, normalize whitespace, preserve case & punctuation as much as possible."""
    if pd.isna(text):
        return ""
    s = str(text)
    s = re.sub(r'http\S+|www\.\S+', '', s)
    s = re.sub(r'\S+@\S+', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    s = handle_emoticons(s)
    return s

# =============================================================================
# HUGGINGFACE API helper (used optionally for seed weak-labeling)
# =============================================================================
class HFSarcasmAPI:
    def __init__(self, api_token: str):
        self.api_token = api_token
        self.headers = {"Authorization": f"Bearer {api_token}"} if api_token and api_token != "your_huggingface_token_here" else {}
        self.url = API_URL
        self.cache = {}  # simple in-memory cache to avoid duplicate calls

    def query_single(self, text: str) -> Optional[Dict[str, Any]]:
        """Query the HF model for a single text. Returns dict or None."""
        if not text:
            return None
        key = text.strip()
        if key in self.cache:
            return self.cache[key]

        payload = {"inputs": key}
        try:
            r = requests.post(self.url, headers=self.headers, json=payload, timeout=30)
            r.raise_for_status()
            resp = r.json()
            # expected structure: [{'label': 'LABEL', 'score': 0.98}, ...] or list-wrapped
            self.cache[key] = resp
            return resp
        except Exception as e:
            logging.warning(f"HF API call failed for text (len={len(text)}): {e}")
            return None

    def query_batch(self, texts: List[str]) -> List[Optional[Any]]:
        """Query up to batch_size texts individually (API expects single inputs in many endpoints)."""
        results = []
        for t in texts:
            res = self.query_single(t)
            results.append(res)
            time.sleep(API_DELAY)
        return results

# =============================================================================
# HEURISTIC FALLBACK (improved)
# =============================================================================
def heuristic_label(text: str) -> Dict[str, Any]:
    """Return a heuristic label dict with keys: 'sarcasm' (bool) and 'confidence' (0..1)."""
    txt = (text or "").lower()
    sarcastic_keywords = ['sarcastic', 'ironic', 'of course', 'sure', 'yeah right', 'just what i needed', 'big surprise', 'wow']
    positive = ['excellent','amazing','perfect','great','love','fantastic']
    negative = ['terrible','awful','horrible','disappointing','worst','bad']
    # keyword check
    if any(kw in txt for kw in sarcastic_keywords):
        return {'sarcasm': True, 'confidence': 0.8, 'reason': 'keyword'}
    # exaggerated positive short review -> likely sarcastic
    pos_count = sum(1 for w in positive if w in txt)
    if pos_count >= 2 and len(txt.split()) < 10:
        return {'sarcasm': True, 'confidence': 0.65, 'reason': 'exaggeration'}
    # mixed positive + negative -> possible sarcasm
    if any(p in txt for p in positive) and any(n in txt for n in negative):
        return {'sarcasm': True, 'confidence': 0.6, 'reason': 'mixed_sentiment'}
    return {'sarcasm': False, 'confidence': 0.3, 'reason': 'none'}

# =============================================================================
# CORE ML COMPONENTS: vectorize, train, evaluate, save
# =============================================================================
def train_model(X_texts: List[str], y_labels: List[int], persist: bool = True):
    """
    Train a TF-IDF + LogisticRegression model with GridSearchCV for regularization.
    Returns trained model and vectorizer.
    """
    logging.info("Vectorizing text (TF-IDF)...")
    vect = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9, max_features=30000)
    X = vect.fit_transform(X_texts)

    logging.info("Train/test split for model training...")
    X_train, X_val, y_train, y_val = train_test_split(X, y_labels, test_size=0.2, random_state=42, stratify=y_labels)

    logging.info("Setting up LogisticRegression + GridSearchCV for regularization tuning...")
    base = LogisticRegression(max_iter=2000, solver='saga', n_jobs=-1)
    param_grid = {
        'C': [0.01, 0.1, 1, 10],
        'penalty': ['l1', 'l2']
    }
    grid = GridSearchCV(base, param_grid, cv=4, scoring='f1', n_jobs=-1, verbose=0)
    grid.fit(X_train, y_train)

    best = grid.best_estimator_
    logging.info(f"Best params: {grid.best_params_}")

    logging.info("Evaluating on validation set...")
    y_pred = best.predict(X_val)
    logging.info("Validation accuracy: %.4f", accuracy_score(y_val, y_pred))
    logging.info("\n" + classification_report(y_val, y_pred))

    if persist:
        logging.info("Saving model and vectorizer...")
        joblib.dump(best, MODEL_FILE)
        joblib.dump(vect, VECT_FILE)
        logging.info(f"Saved model -> {MODEL_FILE}, vectorizer -> {VECT_FILE}")

    return best, vect

# =============================================================================
# BATCH PROCESSING: labeling pipeline
# =============================================================================
def run_pipeline():
    # 1. Load
    logging.info("Loading dataset...")
    try:
        df = pd.read_excel(INPUT_FILE)
    except Exception as e:
        logging.error(f"Failed to load input file: {e}")
        return
    logging.info("Loaded %d rows.", len(df))

    if SAMPLE_SIZE and SAMPLE_SIZE > 0:
        df = df.head(SAMPLE_SIZE).copy()
        logging.info("Using sample of size %d for development.", SAMPLE_SIZE)

    # Ensure review column exists
    if REVIEW_COLUMN not in df.columns:
        logging.error(f"Review column '{REVIEW_COLUMN}' not found in input file.")
        return

    # 2. Preprocess and language filter
    logging.info("Preprocessing reviews and filtering non-English rows...")
    df['processed_review'] = df[REVIEW_COLUMN].apply(preprocess_for_sarcasm)
    df['is_english'] = df['processed_review'].apply(is_english_reliable)
    pre_count = len(df)
    df = df[df['is_english']].reset_index(drop=True)
    logging.info("Kept %d/%d rows after English filtering.", len(df), pre_count)

    if len(df) == 0:
        logging.error("No English reviews to process after filtering.")
        return

    # 3. If LABEL_COLUMN exists in input, use it. Otherwise optionally seed-label via API.
    if LABEL_COLUMN in df.columns and df[LABEL_COLUMN].notna().sum() > 20:
        logging.info("Found existing label column '%s' with %d non-null labels. Using it for supervised training.",
                     LABEL_COLUMN, df[LABEL_COLUMN].notna().sum())
        # Use existing labels
        labeled_df = df[df[LABEL_COLUMN].notna()].copy()
        labeled_texts = labeled_df['processed_review'].tolist()
        labeled_labels = labeled_df[LABEL_COLUMN].astype(int).tolist()
        model, vect = train_model(labeled_texts, labeled_labels, persist=True)
    else:
        # Optional weak-labeling using API for a small seed set
        if API_TOKEN and API_TOKEN != "your_huggingface_token_here":
            logging.info("No adequate labels found in dataset. Weak-labeling a seed set using HF API (seed size=%d).",
                         SEED_LABEL_COUNT)
            api = HFSarcasmAPI(API_TOKEN)
            # sample random seed set
            seed_df = df.sample(n=min(SEED_LABEL_COUNT, len(df)), random_state=42).copy()
            seed_texts = seed_df['processed_review'].tolist()
            # query API (sequential, cached)
            api_responses = api.query_batch(seed_texts)
            # convert responses to binary labels where possible, fallback heuristic otherwise
            seed_labels = []
            seed_texts_final = []
            for text, resp in zip(seed_texts, api_responses):
                label = None
                if resp:
                    # resp format can vary: sometimes a list like [{'label':'NOT_SARCASM','score':...}] for each input
                    if isinstance(resp, dict):
                        # some APIs return a dict for single input
                        # attempt to parse common structure
                        for k in ['label','score']:
                            pass
                    if isinstance(resp, list):
                        # many HF endpoints return a list-of-labels for a single input
                        # e.g. [{'label':'LABEL','score':0.99}]
                        first = resp[0] if resp else None
                        if isinstance(first, dict) and 'label' in first:
                            lbl = first['label'].lower()
                            score = float(first.get('score', 0.0))
                            # heuristics: label containing 'sarcasm' => True
                            if 'sarcasm' in lbl or 'sarcastic' in lbl:
                                label = 1 if score >= 0.5 else 0
                            else:
                                label = 0 if score >= 0.5 else 0
                if label is None:
                    hf_label = heuristic_label(text)
                    label = 1 if hf_label['sarcasm'] else 0
                seed_labels.append(int(label))
                seed_texts_final.append(text)
            # Train model on seed labels
            if len(seed_texts_final) >= 30:
                model, vect = train_model(seed_texts_final, seed_labels, persist=True)
            else:
                logging.warning("Not enough labeled seed samples to train. Falling back to heuristics-only save.")
                # apply heuristics to whole dataset and save
                df['sarcasm'] = df['processed_review'].apply(lambda t: 1 if heuristic_label(t)['sarcasm'] else 0)
                df['sarcasm_confidence'] = df['processed_review'].apply(lambda t: heuristic_label(t)['confidence'])
                df.to_excel(OUTPUT_FILE, index=False)
                logging.info("Saved heuristic-only results to %s", OUTPUT_FILE)
                return
        else:
            # No token provided: rely on heuristics if no labels
            logging.info("No API token provided and no labels found: using heuristics to label and save. To train a model, provide labels or an API token.")
            df['sarcasm'] = df['processed_review'].apply(lambda t: 1 if heuristic_label(t)['sarcasm'] else 0)
            df['sarcasm_confidence'] = df['processed_review'].apply(lambda t: heuristic_label(t)['confidence'])
            df.to_excel(OUTPUT_FILE, index=False)
            logging.info("Saved heuristic-only results to %s", OUTPUT_FILE)
            return

    # 4. If we have a trained model & vectorizer, perform fast batch inference on the entire df
    if os.path.exists(MODEL_FILE) and os.path.exists(VECT_FILE):
        logging.info("Loading trained model and vectorizer for batch inference...")
        model = joblib.load(MODEL_FILE)
        vect = joblib.load(VECT_FILE)
    else:
        logging.info("Using model and vectorizer in memory (just trained).")

    logging.info("Vectorizing all reviews for fast inference...")
    X_all = vect.transform(df['processed_review'].astype(str).tolist())
    logging.info("Predicting sarcasm labels for %d reviews ...", X_all.shape[0])
    preds = model.predict(X_all)
    probs = None
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_all)[:, 1]
    else:
        # fallback: use decision_function if available and minmax-scale
        if hasattr(model, "decision_function"):
            dec = model.decision_function(X_all)
            # scale to 0..1
            probs = (dec - dec.min()) / (dec.max() - dec.min() + 1e-9)
        else:
            probs = np.ones_like(preds, dtype=float)

    df['sarcasm'] = preds.astype(int)
    df['sarcasm_confidence'] = probs.astype(float)
    df['sarcasm_reason'] = 'model_prediction'

    # 5. Save results
    logging.info("Saving results to: %s", OUTPUT_FILE)
    df.to_excel(OUTPUT_FILE, index=False)
    logging.info("Saved %d rows with sarcasm labels.", len(df))

    # 6. Summary
    n_sarcastic = int(df['sarcasm'].sum())
    logging.info("Sarcastic reviews detected: %d (%.2f%%)", n_sarcastic, 100.0 * n_sarcastic / len(df))

    # show a few sample sarcastic rows
    if n_sarcastic > 0:
        logging.info("Sample sarcastic review(s):")
        samples = df[df['sarcasm'] == 1].head(5)
        for idx, row in samples.iterrows():
            logging.info(" - %s ... (conf=%.2f)", row['processed_review'][:150], row['sarcasm_confidence'])

# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    start = time.time()
    logging.info("Starting optimized batch sarcasm pipeline.")
    run_pipeline()
    logging.info("Total runtime: %.2f seconds", time.time() - start)


Installing required packages...
Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Starting batch sarcasm detection pipeline...
BATCH SARCASM DETECTION PIPELINE
1. Loading data...
   Loaded 6780 reviews from /content/drive/MyDrive/Colab Notebooks/hotels.xlsx
   Using sample of 200 reviews for testing
2. Filtering English reviews...
   Removed 80 non-English reviews
3. Running batch sarcasm detection...
Processing 120 reviews for sarcasm using batch size 5...
Processing batch 1/24 (5 reviews)
Processing batch 2/24 (5 reviews)
Processing batch 3/24 (5 reviews)
Processing batch 4/24 (5 reviews)
Processing batch 5/24 (5 reviews)
Processing batch 6/24 (5 reviews)
Processing batch 7/24 (5 reviews)
Processing batch 8/24 (5 reviews)
Processing batch 9/24 (5 reviews)
Processing batch 10/24 (5 reviews)
Processing batch 11/24 (5 reviews)
Processing batch 12/24 (5 reviews)
Processing batch 13/24 